### DESCRIPTION (inspired by Leetcode.com)
medium

You are given an m x n matrix of non-negative integers representing a grid of land, where rain falls on every cell. Each value in the grid represents the height of that piece of land.

The Pacific Ocean touches the left and top edges of the matrix, while the Atlantic Ocean touches the right and bottom edges. Water can only flow from a cell to its neighboring cells directly north, south, east, or west, but only if the height of the neighboring cell is equal to or lower than the current cell.

Write a function to return a list of grid coordinates (i, j) where water can flow to both the Pacific and Atlantic Oceans. Water can flow from all cells directly adjacent to the ocean into that ocean.

Example 1:

Input:

```
grid = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]
```
Output:
```
[
    [0, 2],
    [1, 2],
    [2, 0],
    [2, 1],
    [2, 2]
]
```

In [1]:
class Solution:
    def pacific_atlantic_flow(self, grid: list[list[int]]) -> list[list[int]]:
        if not grid or not grid[0]:
            return []
        rows, cols = len(grid), len(grid[0])
        pacific_flow, atlantic_flow = set(), set()
        result = []

        # flow to pacific
        for r in range(rows):
            for c in range(cols):
                if r == 0 or c == 0:
                    pacific_flow.add((r,c))
                elif ((r-1,c) in pacific_flow and grid[r][c] >= grid[r-1][c]) or \
                    ((r,c-1) in pacific_flow and grid[r][c] >= grid[r][c-1]):
                    pacific_flow.add((r,c))

        # flow to atlantic
        for r in range(rows-1,-1,-1):
            for c in range(cols-1,-1,-1):
                if r == rows-1 or c == cols-1:
                    atlantic_flow.add((r,c))
                elif ((r+1,c) in atlantic_flow and grid[r][c] >= grid[r+1][c]) or \
                    ((r,c+1) in atlantic_flow and grid[r][c] >= grid[r][c+1]):
                    atlantic_flow.add((r,c))
        
        for r in range(rows):
            for c in range(cols):
                if (r, c) in pacific_flow and (r,c) in atlantic_flow:
                    result.append([r,c])

        return result        

### Feedback

Your solution passes the tests and uses an efficient O(mn) time approach with O(mn) set space. The directional scans correctly propagate reachability through cells whose heights are nondecreasing toward the ocean. 

A useful interview caveat: unlike DFS/BFS, this works only because each scan’s traversal order guarantees that relevant north/west (Pacific) or south/east (Atlantic) states have already been processed. It is less obviously general for arbitrary paths, so be prepared to justify that invariant—or use multi-source BFS/DFS, which naturally handles all four directions. 

Minor readability improvement: add spaces in tuples such as (r, c) and consider a set comprehension for the final intersection. Also, the return annotation says List[List[int]], while the prompt describes coordinates as tuples; your returned lists are consistent with the reference output.

In [39]:
class Solution:
    def pacific_atlantic_flow(self, grid: list[list[int]]) -> list[list[int]]:
        if not grid or not grid[0]:
            return []
        rows, cols = len(grid), len(grid[0])
        pacific_reachable, atlantic_reachable = set(), set()
        directions = [(0,-1), (-1,0), (0,1), (1,0)]

        def dfs(r: int, c: int, reachable: set):
            reachable.add((r,c))
            for dr, dc in directions:
                r_ = r + dr
                c_ = c + dc
                if 0 <= r_ < rows and 0 <= c_ < cols:
                    if (r_, c_) not in reachable and grid[r_][c_] >= grid[r][c]:
                        dfs(r_, c_, reachable)

        for r in range(rows):
            dfs(r, 0, pacific_reachable)
            dfs(r, cols-1, atlantic_reachable)

        for c in range(cols):
            dfs(0, c, pacific_reachable)
            dfs(rows-1, c, atlantic_reachable)

        result = []
        for r in range(rows):
            for c in range(cols):
                if (r, c) in pacific_reachable and (r,c) in atlantic_reachable:
                    result.append([r,c])
        
        return result     

        # can use this if order doesn't matter
        #return [[r,c] for r,c in atlantic_reachable & pacific_reachable]

In [2]:
from typing import Callable

class Test:  
    def __init__(self, input: list[list[int]], result: list[list[int]]):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[list[list[int]]], list[list[int]]]):
    for test in tests:
        org_input = str(test.input)
        result = func(test.input)
        if result == test.expected_result:
            print(f"Test passed for {org_input}")
        else:
            print(f"Test failed for {org_input}. Expected: {test.expected_result}, Actual: {result}")

In [40]:
tests = [
    Test([],[]),
    Test([[1]],[[0,0]]),
    Test([[1,2]],[[0,0],[0,1]]),
    Test([[2,1]],[[0,0],[0,1]]),
    Test([[1],[2]],[[0,0],[1,0]]),
    Test([[2],[1]],[[0,0],[1,0]]),
    Test([[1, 2, 3],[4, 5, 6],[7, 8, 9]],[[0, 2],[1, 2],[2, 0],[2, 1],[2, 2]])
]

run_tests(tests, Solution().pacific_atlantic_flow)

Test passed for []
Test passed for [[1]]
Test passed for [[1, 2]]
Test passed for [[2, 1]]
Test passed for [[1], [2]]
Test passed for [[2], [1]]
Test passed for [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
